In [ ]:
!pip install -q transformers sentencepiece

from google.colab import files
uploaded = files.upload()   # selecione o fato_review.csv

Saving fato_review.csv to fato_review.csv


In [ ]:
import pandas as pd

df = pd.read_csv("fato_review.csv", sep=";", encoding="utf-8-sig")

# O mesmo review_id aparece em vários produtos com o mesmo texto,
# então classificamos cada comentário uma única vez
df = df.drop_duplicates(subset="review_id").copy()

df["texto"] = (
    df["review_title"].fillna("") + ". " + df["review_content"].fillna("")
).str.strip(". ").str.strip()

df = df[df["texto"] != ""]
print(len(df), "comentários únicos para classificar")

9079 comentários únicos para classificar


In [ ]:
import torch
from transformers import pipeline
from tqdm.auto import tqdm

device = 0 if torch.cuda.is_available() else -1

# Rótulos em inglês (o modelo entende melhor) -> nomes em português para o dashboard
temas = {
    "product quality": "Qualidade",
    "price and value for money": "Preço/custo-benefício",
    "durability": "Durabilidade",
    "battery and charging": "Bateria",
    "delivery and packaging": "Entrega/embalagem",
    "ease of use": "Facilidade de uso",
    "design and appearance": "Design",
}

clf_tema = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0",
    device=device,
)
clf_sent = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=device,
)

textos = df["texto"].tolist()
res_tema, res_sent = [], []
lote = 256

for i in tqdm(range(0, len(textos), lote)):
    bloco = textos[i:i + lote]
    res_tema += clf_tema(
        bloco,
        candidate_labels=list(temas.keys()),
        hypothesis_template="This review is about {}.",
        batch_size=32,
    )
    res_sent += clf_sent(bloco, batch_size=32, truncation=True, max_length=512)

# Tema com baixa confiança vira "Outros"
df["tema"] = [
    temas[r["labels"][0]] if r["scores"][0] >= 0.35 else "Outros"
    for r in res_tema
]
df["confianca_tema"] = [round(r["scores"][0], 3) for r in res_tema]
df["sentimento"] = [
    {"positive": "Positivo", "neutral": "Neutro", "negative": "Negativo"}[r["label"]]
    for r in res_sent
]

print(df["tema"].value_counts())
print(df["sentimento"].value_counts())

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

... (14 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  0%|          | 0/36 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


tema
Qualidade                7807
Facilidade de uso         487
Preço/custo-benefício     401
Bateria                   218
Outros                     57
Design                     54
Durabilidade               39
Entrega/embalagem          16
Name: count, dtype: int64
sentimento
Positivo    6665
Neutro      1232
Negativo    1182
Name: count, dtype: int64


In [ ]:
saida = df[["review_id", "tema", "confianca_tema", "sentimento"]]
saida.to_csv("review_temas.csv", sep=";", decimal=",", index=False, encoding="utf-8-sig")
files.download("review_temas.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch
print(torch.cuda.is_available())

True
